# 6주차 추가 실습 · PyTorch로 CNN 학습하기

**이번 실습:** 공개 이미지 데이터 CIFAR-10으로 CNN을 직접 만들어 학습하고, 같은 모델을 CPU와 GPU에서 돌려 시간을 비교합니다. 마지막에 순서 데이터용 RNN의 입력 모양을 확인합니다.
**준비:** 5주차 뉴런의 학습 원리, 6주차 신경망 검증

이 파일 하나를 위에서부터 순서대로 실행합니다. `#` 뒤의 한국어 주석을 한 줄씩 읽고 실행해 주세요.

**이 실습은 Colab에서 진행합니다.** 실행 전에 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다. 전체 데이터로 학습하고 CPU와 GPU의 속도 차이를 직접 봅니다.

VS Code에서도 CPU로 축소 실행(학습용 1만 장, 2바퀴)은 됩니다. 이 경우 커널은 **Python (semi-ai)**를 고르고, 터미널에서 `conda activate semi-ai` 후 `pip install torchvision==0.15.2`를 한 번 실행합니다. GPU 비교 셀은 CPU 시간만 나옵니다.

데이터는 처음 한 번 내려받습니다(약 170MB). 한 번 받으면 `data/cifar10` 폴더에 남습니다.

## 0. 실습 폴더 준비

VS Code와 Colab 어디서 열어도 `data` 폴더를 찾도록 실습 폴더 위치를 정합니다. Colab에서는 처음 한 번 실습 자료를 내려받습니다.

In [ ]:
import sys, subprocess  # 실행 환경 확인과 명령 실행 도구입니다.
from pathlib import Path  # 파일 위치 도구입니다.
if "google.colab" in sys.modules:  # Colab에서 실행 중이라면
    folder = Path("/content/semiconductor_ai_beginner")  # 실습 자료를 둘 위치입니다.
    if not folder.exists():  # 아직 내려받지 않았다면
        subprocess.run(["git", "clone", "-q", "https://github.com/SeokJH-TAs/semiconductor_ai_beginner.git", str(folder)], check=True)  # 실습 자료 전체를 내려받습니다.
else:  # VS Code 등 내 컴퓨터에서 실행 중이라면
    folder = Path.cwd()  # 현재 폴더입니다.
    if folder.name == "notebooks":  # 노트북 폴더 안이라면
        folder = folder.parent  # 실습 전체 폴더로 올라갑니다.
print(folder)  # 실습 폴더 위치를 확인합니다.

## 1. 도구 불러오기와 장치 확인

`torch`는 텐서 계산과 자동 미분, `nn`은 신경망 층과 손실 함수, `torchvision`은 이미지 데이터셋을 내려받는 도구입니다.

`device`는 계산을 어디서 할지 정하는 이름표입니다. GPU가 있으면 `cuda`, 없으면 `cpu`입니다. 이 값 하나로 뒤의 모든 코드가 CPU·GPU 어디서든 돌아갑니다.

In [ ]:
import time  # 시간을 재는 도구입니다.
import numpy as np  # 배열 계산 도구입니다.
import matplotlib.pyplot as plt  # 그림 도구입니다.
plt.rcParams["font.size"] = 14  # 발표 화면에서도 글자를 읽기 쉽게 키웁니다.
import torch  # 텐서와 자동 미분 도구입니다.
from torch import nn  # 신경망 층과 손실 함수 모음입니다.
from torch.utils.data import DataLoader, Subset  # 배치로 꺼내는 도구와 일부만 고르는 도구입니다.
try:  # torchvision이 설치되어 있는지 확인합니다.
    from torchvision import datasets, transforms  # 공개 이미지 데이터셋과 변환 도구입니다.
except ImportError:  # 없다면
    raise SystemExit("torchvision이 없습니다. 터미널에서 conda activate semi-ai 후 pip install torchvision==0.15.2 를 실행하고 커널을 다시 시작하세요.")  # 설치 방법을 안내하고 멈춥니다.

device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU가 있으면 cuda, 없으면 cpu입니다.
print("PyTorch:", torch.__version__)  # 설치된 버전입니다.
print("장치:", device)  # 이번 실습이 돌아갈 장치입니다.
if device == "cuda":  # GPU가 있다면
    print("GPU 이름:", torch.cuda.get_device_name(0))  # 어떤 GPU인지 봅니다.
else:  # GPU가 없다면
    print("GPU가 없습니다. Colab이라면 런타임 → 런타임 유형 변경 → T4 GPU를 선택하세요.")  # 안내입니다.

## 2. CIFAR-10 내려받기

CIFAR-10은 32×32 컬러 사진 6만 장을 10가지(비행기, 자동차, 새, 고양이, 사슴, 개, 개구리, 말, 배, 트럭)로 나눈 공개 데이터입니다. 학습용 5만 장, 시험용 1만 장으로 처음부터 나뉘어 있습니다.

3주차의 공정 표는 한 행이 숫자 3개였습니다. 여기서는 한 장이 **3×32×32 = 3,072개** 숫자입니다. 이렇게 차원이 커지면 계산량이 늘어나고, 그래서 GPU가 필요해집니다.

`ToTensor()`는 사진을 0~1 사이 숫자의 텐서로 바꿉니다. 모양은 `(채널, 높이, 너비)` 순서입니다.

원본 서버(토론토대)가 느릴 때가 있어, 같은 파일을 수업 저장소에 올려 두고 거기서 받습니다. 파일 검증(md5)은 torchvision이 그대로 합니다.

In [ ]:
to_tensor = transforms.ToTensor()  # 사진을 0~1 범위의 텐서로 바꾸는 변환입니다.
data_root = folder / "data" / "cifar10"  # 내려받은 파일을 둘 위치입니다.
datasets.CIFAR10.url = "https://github.com/SeokJH-TAs/semiconductor_ai_beginner/releases/download/data/cifar-10-python.tar.gz"  # 원본 서버가 느려서 수업용 저장소의 같은 파일을 받습니다.
train_data = datasets.CIFAR10(root=data_root, train=True, download=True, transform=to_tensor)  # 학습용 5만 장입니다.
test_data = datasets.CIFAR10(root=data_root, train=False, download=True, transform=to_tensor)  # 시험용 1만 장입니다.
class_names = train_data.classes  # 10가지 이름 목록입니다.
image, label = train_data[0]  # 첫 번째 사진과 정답 번호입니다.
print("학습 장수:", len(train_data), "시험 장수:", len(test_data))  # 50000, 10000입니다.
print("한 장의 모양:", image.shape, "숫자 개수:", image.numel())  # (3, 32, 32), 3072입니다.
print("정답 번호:", label, "이름:", class_names[label])  # 숫자 라벨과 이름입니다.
print("종류:", class_names)  # 10가지 이름입니다.

### 사진 몇 장 보기

숫자 격자가 실제로 어떤 사진인지 봅니다. `permute(1, 2, 0)`은 `(채널, 높이, 너비)`를 그림 도구가 원하는 `(높이, 너비, 채널)`로 축 순서를 바꿉니다.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))  # 가로로 사진 6장을 놓을 자리입니다.
for i, axis in enumerate(axes):  # 자리마다 하나씩
    image, label = train_data[i]  # i번째 사진과 정답입니다.
    axis.imshow(image.permute(1, 2, 0))  # 축 순서를 바꿔 사진으로 그립니다.
    axis.set_title(class_names[label])  # 정답 이름을 제목으로 붙입니다.
    axis.axis("off")  # 눈금은 숨깁니다.
plt.tight_layout()  # 여백을 맞춥니다.
plt.show()  # 사진을 보여 줍니다.

## 3. DataLoader로 배치 꺼내기

`DataLoader`는 사진과 정답을 **배치** 단위로 꺼내 줍니다. `batch_size=128`은 한 번에 128장, `shuffle=True`는 학습 순서를 섞는다는 뜻입니다. 5만 장이면 한 바퀴(epoch)에 391배치입니다.

GPU가 없는 컴퓨터에서는 학습용 1만 장만 골라 씁니다(`Subset`). 흐름은 같고 시간이 짧아집니다.

In [ ]:
torch.manual_seed(42)  # 섞는 순서와 시작 가중치를 재현하기 위한 씨앗입니다.
if device == "cpu":  # GPU가 없다면
    train_data = Subset(train_data, range(min(10000, len(train_data))))  # 앞 1만 장만 사용해 시간을 줄입니다.
    print("CPU 실행: 학습용 1만 장만 사용합니다.")  # 안내입니다.
train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=0)  # 학습용 배치입니다.
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=0)  # 시험용 배치입니다. 섞지 않습니다.
batch_x, batch_y = next(iter(train_loader))  # 첫 배치 하나를 꺼냅니다.
print("배치 수:", len(train_loader))  # 한 바퀴에 꺼내는 횟수입니다.
print("배치 입력 모양:", batch_x.shape)  # (128, 3, 32, 32)입니다.
print("배치 정답 모양:", batch_y.shape)  # (128,)입니다. 정답은 번호 하나입니다.
print("첫 정답 다섯 개:", batch_y[:5].tolist())  # 0~9 사이 번호입니다.

## 4. CNN을 nn.Module로 만들기

`class`는 모델 설계도입니다. `nn.Module`을 물려받아 층을 등록하고, `__init__`에서 층을 준비하고, `forward`에서 입력이 출력까지 흐르는 계산을 적습니다. 6주차 `SmallNetwork`와 같은 틀이고, 안에 들어가는 층만 다릅니다.

| 층 | 하는 일 | 모양 변화 (한 장 기준) |
|---|---|---|
| `Conv2d(3, 32, 3, padding=1)` | 3×3 창을 사진 위로 밀며 무늬 32종류를 찾습니다 | (3,32,32) → (32,32,32) |
| `ReLU` | 음수를 0으로 바꿉니다 | 모양 그대로 |
| `MaxPool2d(2)` | 2×2마다 가장 큰 값만 남겨 크기를 절반으로 줄입니다 | (32,32,32) → (32,16,16) |
| `Conv2d(32, 64, 3, padding=1)` + `ReLU` + `MaxPool2d(2)` | 한 번 더 | (32,16,16) → (64,8,8) |
| `Flatten` | 격자를 한 줄로 폅니다 | (64,8,8) → 4096 |
| `Linear(4096, 10)` | 10가지 점수를 냅니다 | 4096 → 10 |

마지막 출력 10개는 각 종류에 대한 점수(logit)입니다. 가장 큰 점수의 번호가 예측입니다. 분류에서는 출력층에 sigmoid를 붙이지 않고, 손실 함수(`CrossEntropyLoss`)가 점수를 확률로 바꿔 채점합니다.

In [ ]:
class SmallCNN(nn.Module):  # 사진을 입력받는 작은 합성곱 신경망 설계도입니다.
    def __init__(self):  # 모델을 만들 때 한 번 실행합니다.
        super().__init__()  # nn.Module의 기본 준비를 합니다.
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # 채널 3을 32가지 무늬 지도로 바꿉니다.
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # 32가지를 64가지로 늘립니다.
        self.pool = nn.MaxPool2d(2)  # 가로·세로를 절반으로 줄입니다.
        self.activation = nn.ReLU()  # 음수를 0으로 바꾸는 비선형 변환입니다.
        self.flatten = nn.Flatten()  # 격자를 한 줄로 폅니다.
        self.output = nn.Linear(64 * 8 * 8, 10)  # 4096개 숫자로 10가지 점수를 만듭니다.

    def forward(self, x):  # 입력 x가 출력까지 흐르는 계산입니다.
        x = self.pool(self.activation(self.conv1(x)))  # (3,32,32) → (32,16,16)
        x = self.pool(self.activation(self.conv2(x)))  # (32,16,16) → (64,8,8)
        x = self.flatten(x)  # (64,8,8) → 4096
        return self.output(x)  # 4096 → 10가지 점수

### 층을 지나며 바뀌는 모양 확인

학습 전 모델에 첫 배치를 넣고 각 층의 출력 모양을 찍어 봅니다. 모양이 표와 같으면 설계도가 맞게 연결된 것입니다. `parameters()`로 optimizer가 바꿀 숫자가 얼마나 되는지도 셉니다.

In [ ]:
torch.manual_seed(42)  # 시작 가중치를 고정합니다.
demo = SmallCNN()  # 설계도로 모델 하나를 만듭니다.
x = batch_x  # 3번에서 꺼낸 첫 배치입니다.
print("입력:", x.shape)  # (128, 3, 32, 32)
x = demo.pool(demo.activation(demo.conv1(x)))  # 첫 합성곱 블록을 통과합니다.
print("conv1 + pool:", x.shape)  # (128, 32, 16, 16)
x = demo.pool(demo.activation(demo.conv2(x)))  # 둘째 합성곱 블록을 통과합니다.
print("conv2 + pool:", x.shape)  # (128, 64, 8, 8)
x = demo.flatten(x)  # 한 줄로 폅니다.
print("flatten:", x.shape)  # (128, 4096)
x = demo.output(x)  # 10가지 점수입니다.
print("output:", x.shape)  # (128, 10)
print("첫 사진의 점수 10개:", x[0].detach().numpy().round(2))  # 아직 학습 전이라 의미 없는 숫자입니다.
total = sum(p.numel() for p in demo.parameters())  # 가중치와 편향 개수를 모두 더합니다.
print("학습할 숫자 개수:", total)  # optimizer가 바꿀 숫자의 수입니다.

## 5. 한 배치에서 손실·역전파·갱신 보기

5주차에서 손으로 했던 순서를 PyTorch가 대신합니다. `loss.backward()`가 모든 가중치의 기울기를 채우고, `optimizer.step()`이 그 기울기로 가중치를 바꿉니다. `zero_grad()`는 이전 기울기를 비웁니다. 세 줄의 순서가 바뀌면 안 됩니다.

여기서는 모델과 배치를 `.to(device)`로 같은 장치에 올립니다. 모델은 GPU에 있는데 데이터가 CPU에 있으면 오류가 납니다.

In [ ]:
loss_function = nn.CrossEntropyLoss()  # 10가지 점수와 정답 번호를 비교하는 분류 손실입니다.
demo = demo.to(device)  # 모델을 장치에 올립니다.
optimizer = torch.optim.SGD(demo.parameters(), lr=0.05, momentum=0.9)  # 가중치를 바꿀 도구입니다.
x, y = batch_x.to(device), batch_y.to(device)  # 데이터도 같은 장치에 올립니다.
optimizer.zero_grad()  # 지난 기울기를 비웁니다.
prediction = demo(x)  # 현재 가중치로 점수를 냅니다.
loss = loss_function(prediction, y)  # 정답과 비교한 손실입니다.
print("배치 손실:", round(loss.item(), 4))  # 10가지 중 무작위로 찍으면 약 2.3(=ln 10)입니다.
before = demo.conv1.weight[0, 0, 0].detach().cpu().numpy().copy()  # 첫 필터 첫 줄의 사본입니다.
print("갱신 전 conv1 가중치 일부:", before.round(6))  # 갱신 전 값입니다.
loss.backward()  # 모든 가중치의 기울기를 계산합니다.
print("기울기:", demo.conv1.weight.grad[0, 0, 0].cpu().numpy().round(6))  # 같은 위치의 기울기입니다.
optimizer.step()  # 기울기 방향으로 가중치를 바꿉니다.
after = demo.conv1.weight[0, 0, 0].detach().cpu().numpy()  # 갱신 후 값입니다.
print("갱신 후 conv1 가중치 일부:", after.round(6))  # 조금 바뀌었습니다.
print("변화량(= -lr × 기울기):", (after - before).round(6))  # 보폭 0.05에 기울기를 곱한 만큼 움직였습니다.

## 6. CPU와 GPU 속도 비교

같은 모델, 같은 배치를 CPU와 GPU에서 각각 **50배치만** 학습시켜 배치당 시간을 잽니다. 전체를 다 돌리지 않아도 1 epoch에 걸릴 시간을 계산할 수 있습니다.

- 처음 몇 배치는 준비 작업 때문에 느리므로 5배치를 먼저 돌리고(warm-up) 그다음부터 잽니다.
- GPU는 계산을 뒤로 미뤄 두고 다음 줄로 넘어가기 때문에, `torch.cuda.synchronize()`로 계산이 끝나길 기다린 뒤 시간을 읽어야 정확합니다.
- GPU가 없는 컴퓨터에서는 CPU 결과만 나옵니다.

In [ ]:
def measure(device_name, max_batches=50, warm_up=5):  # 한 장치에서 배치당 학습 시간을 재는 함수입니다.
    torch.manual_seed(42)  # 같은 시작 가중치입니다.
    model = SmallCNN().to(device_name)  # 모델을 해당 장치에 올립니다.
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)  # 갱신 도구입니다.
    done = 0  # 잰 배치 수입니다.
    for x, y in train_loader:  # 배치를 하나씩 꺼냅니다.
        x, y = x.to(device_name), y.to(device_name)  # 같은 장치에 올립니다.
        if done == warm_up:  # 준비 배치가 끝났으면
            if device_name == "cuda":  # GPU라면
                torch.cuda.synchronize()  # 밀린 계산이 끝나길 기다립니다.
            start = time.time()  # 여기서부터 시간을 잽니다.
        optimizer.zero_grad()  # 기울기 비우기
        loss = loss_function(model(x), y)  # 예측과 손실
        loss.backward()  # 역전파
        optimizer.step()  # 갱신
        done += 1  # 한 배치 끝
        if done >= warm_up + max_batches:  # 정한 수만큼 쟀으면
            break  # 멈춥니다.
    if device_name == "cuda":  # GPU라면
        torch.cuda.synchronize()  # 마지막 계산까지 기다립니다.
    per_batch = (time.time() - start) / max_batches  # 배치당 시간입니다.
    return per_batch  # 초 단위입니다.

full_epoch_batches = 50000 // 128 + 1  # 전체 5만 장 기준 1 epoch의 배치 수(391)입니다.
cpu_time = measure("cpu")  # CPU에서 잽니다.
print(f"CPU: 배치당 {cpu_time*1000:.1f}ms → 5만 장 1 epoch 예상 {cpu_time*full_epoch_batches/60:.1f}분")  # CPU 예상입니다.
if device == "cuda":  # GPU가 있다면
    gpu_time = measure("cuda")  # GPU에서도 잽니다.
    print(f"GPU: 배치당 {gpu_time*1000:.1f}ms → 5만 장 1 epoch 예상 {gpu_time*full_epoch_batches:.1f}초")  # GPU 예상입니다.
    print(f"GPU가 약 {cpu_time/gpu_time:.0f}배 빠릅니다.")  # 배율입니다.
else:  # GPU가 없다면
    print("GPU가 없어 CPU만 쟀습니다. Colab에서 T4 GPU를 켜고 다시 실행해 보세요.")  # 안내입니다.

## 7. 학습 함수 만들고 실행하기

`train_model`은 어떤 모델이든 받아서 정해진 바퀴(epoch)만큼 학습하고, 바퀴마다 평균 손실과 시험 정확도를 기록합니다. CNN과 뒤의 RNN에 같은 함수를 씁니다.

`evaluate`는 시험 1만 장을 넣어 맞힌 비율을 셉니다. 평가할 때는 `eval()`과 `no_grad()`로 학습 기록을 끕니다. 시험 데이터는 가중치 갱신에 쓰지 않습니다.

GPU에서는 5바퀴, CPU에서는 2바퀴만 돕니다.

In [ ]:
def evaluate(model):  # 시험 데이터로 정확도를 재는 함수입니다.
    model.eval()  # 평가 모드입니다.
    correct = 0  # 맞힌 수입니다.
    with torch.no_grad():  # 평가에는 기울기가 필요 없습니다.
        for x, y in test_loader:  # 시험 배치를 하나씩
            x, y = x.to(device), y.to(device)  # 장치에 올립니다.
            predicted = model(x).argmax(dim=1)  # 점수가 가장 큰 번호가 예측입니다.
            correct += (predicted == y).sum().item()  # 맞힌 개수를 더합니다.
    model.train()  # 다시 학습 모드로 돌립니다.
    return correct / len(test_data)  # 맞힌 비율입니다.

def train_model(model, epochs, lr=0.05):  # 모델을 받아 학습하고 기록을 돌려주는 함수입니다.
    model = model.to(device)  # 장치에 올립니다.
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)  # 갱신 도구입니다.
    history = []  # (바퀴, 손실, 정확도) 기록입니다.
    print(f"학습 전 시험 정확도: {evaluate(model):.3f}")  # 10가지 중 찍으면 약 0.1입니다.
    for epoch in range(epochs):  # 정해진 바퀴만큼
        start = time.time()  # 한 바퀴 시간을 잽니다.
        total_loss = 0.0  # 손실 합입니다.
        for x, y in train_loader:  # 배치를 하나씩
            x, y = x.to(device), y.to(device)  # 장치에 올립니다.
            optimizer.zero_grad()  # 기울기 비우기
            loss = loss_function(model(x), y)  # 예측과 손실
            loss.backward()  # 역전파
            optimizer.step()  # 갱신
            total_loss += loss.item() * len(x)  # 행 수를 곱해 더합니다.
        epoch_loss = total_loss / len(train_data)  # 한 장당 평균 손실입니다.
        accuracy = evaluate(model)  # 이번 바퀴 끝의 시험 정확도입니다.
        history.append((epoch + 1, epoch_loss, accuracy))  # 기록합니다.
        print(f"epoch {epoch+1}: 손실 {epoch_loss:.3f}, 시험 정확도 {accuracy:.3f}, {time.time()-start:.0f}초")  # 진행 상황입니다.
    return model, history  # 학습된 모델과 기록입니다.

epochs = 5 if device == "cuda" else 2  # GPU면 5바퀴, CPU면 2바퀴입니다.
torch.manual_seed(42)  # 시작 가중치를 고정합니다.
cnn, cnn_history = train_model(SmallCNN(), epochs)  # CNN을 학습합니다.

## 8. 손실 곡선과 정확도

바퀴마다 손실이 내려가고 정확도가 올라가는지 봅니다. 손실은 학습 데이터, 정확도는 시험 데이터 기준입니다.

In [ ]:
epochs_done = [h[0] for h in cnn_history]  # 바퀴 번호입니다.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # 그림 두 장입니다.
axes[0].plot(epochs_done, [h[1] for h in cnn_history], marker="o")  # 손실입니다.
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Training loss"); axes[0].set_title("CNN loss")  # 축 이름입니다.
axes[1].plot(epochs_done, [h[2] for h in cnn_history], marker="o")  # 정확도입니다.
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Test accuracy"); axes[1].set_title("CNN accuracy")  # 축 이름입니다.
axes[1].set_ylim(0, 1)  # 정확도는 0~1입니다.
plt.tight_layout()  # 여백을 맞춥니다.
plt.show()  # 그림을 보여 줍니다.

### 예측 결과 보기

시험 사진 8장에 대해 모델의 예측과 실제 정답을 나란히 봅니다. 틀린 것은 제목이 빨간색입니다.

In [ ]:
cnn.eval()  # 평가 모드입니다.
x, y = next(iter(test_loader))  # 시험 첫 배치입니다.
with torch.no_grad():  # 기울기 없이
    predicted = cnn(x.to(device)).argmax(dim=1).cpu()  # 예측 번호입니다.
fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))  # 사진 8장 자리입니다.
for i, axis in enumerate(axes):  # 자리마다
    axis.imshow(x[i].permute(1, 2, 0))  # 사진을 그립니다.
    color = "black" if predicted[i] == y[i] else "red"  # 맞으면 검정, 틀리면 빨강입니다.
    axis.set_title(f"{class_names[predicted[i]]}\n({class_names[y[i]]})", color=color, fontsize=10)  # 예측(정답)입니다.
    axis.axis("off")  # 눈금은 숨깁니다.
plt.tight_layout()  # 여백을 맞춥니다.
plt.show()  # 보여 줍니다.

## 9. RNN의 입력 모양 읽기

CNN은 사진처럼 **이웃 관계**가 있는 데이터, RNN은 센서 기록처럼 **시간 순서**가 있는 데이터에 맞는 구조입니다. 사진은 순서 데이터가 아니므로 여기서는 작은 순서 데이터를 직접 만들어 RNN이 입력을 어떻게 읽는지 모양만 확인합니다. 학습은 CNN과 같은 `train_model`로 할 수 있습니다.

[RNN 공식 문서](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html): `batch_first=True`이면 입력은 `(N, L, F)`입니다. N은 기록 묶음 수, L은 시간 길이, F는 시점마다의 특징 수입니다. 아래는 **공정 두 번을 각각 네 시점에 걸쳐, 센서 두 개로 측정했다고 가정한 합성 예시**입니다. 기존 process.csv의 독립 행을 시간 순서로 바꿔 부르지 않습니다.

`output[:, -1, :]`는 모든 묶음에서 마지막 시점의 은닉 특징을 꺼냅니다. 마지막 시점의 결과 하나를 예측하는 작업을 가정한 선택입니다. 매 시점 예측이 목표라면 출력 선택과 정답 모양도 달라집니다. `batch_first`는 여기서 입력·output의 축 순서를 정하며 hidden의 모양은 여전히 `(층 수, N, 은닉 수)`입니다.

In [ ]:
sequence = torch.tensor([  # 시간 기록 두 묶음을 텐서로 만듭니다.
    [[0.0, 0.1], [0.2, 0.2], [0.4, 0.3], [0.6, 0.4]],  # 첫 공정의 네 시점·두 센서입니다.
    [[0.6, 0.4], [0.4, 0.3], [0.2, 0.2], [0.0, 0.1]],  # 둘째 공정의 네 시점·두 센서입니다.
])  # 전체 모양은 (2, 4, 2)입니다.
rnn = nn.RNN(input_size=2, hidden_size=4, batch_first=True)  # 시점마다 센서 2개를 읽습니다.
sequence_output, last_hidden = rnn(sequence)  # 모든 시점의 특징과 마지막 은닉 상태입니다.
last_feature = sequence_output[:, -1, :]  # 마지막 시점의 특징 4개입니다.
rnn_head = nn.Linear(4, 1)  # 마지막 특징으로 결과 하나를 만듭니다.
rnn_output = rnn_head(last_feature)  # 아직 학습하지 않은 출력입니다.
print("입력:", sequence.shape, "시점별 출력:", sequence_output.shape)  # (2, 4, 2), (2, 4, 4)입니다.
print("마지막 은닉:", last_hidden.shape, "마지막 특징:", last_feature.shape)  # (1, 2, 4), (2, 4)입니다.
print("최종 출력:", rnn_output.shape)  # 공정 두 번에 결과 하나씩, (2, 1)입니다.
print("미학습 출력:", rnn_output.detach())  # 학습하지 않아 의미 있는 예측값은 아닙니다.

In [ ]:
plt.figure(figsize=(6, 4))  # 첫 공정 묶음의 센서 기록입니다.
plt.plot(range(4), sequence[0, :, 0], marker="o", label="Sensor 1")  # 첫 센서의 시간 순서입니다.
plt.plot(range(4), sequence[0, :, 1], marker="o", label="Sensor 2")  # 둘째 센서의 시간 순서입니다.
plt.xlabel("Time step")  # 네 시점입니다.
plt.ylabel("Synthetic sensor value")  # 합성 숫자입니다.
plt.legend()  # 센서를 구분합니다.
plt.tight_layout()  # 여백을 맞춥니다.
plt.show()  # 시간 축의 의미를 확인합니다.

## 10. 역할을 말로 정리하기

| 코드 | 맡은 일 |
|---|---|
| `datasets.CIFAR10` | 공개 데이터 내려받기. 한 장 = (3, 32, 32) 숫자 격자 |
| `DataLoader` | 사진·정답 짝을 배치로 꺼내기 |
| `device`, `.to(device)` | 모델과 데이터를 CPU 또는 GPU에 올리기 |
| `class SmallCNN(nn.Module)` | Conv2d → ReLU → MaxPool을 두 번 거쳐 10가지 점수 |
| `nn.RNN` | 시간 순서가 있는 입력 (묶음, 시간, 특징)을 한 시점씩 읽기 |
| `CrossEntropyLoss` | 10가지 점수와 정답 번호의 차이를 숫자로 |
| `zero_grad` → `backward` → `step` | 기울기 비우기 → 계산 → 가중치 갱신 |
| `train_model`, `evaluate` | 학습 반복과 시험 채점 |

**GPU를 쓰는 이유:** 6번에서 잰 배치당 시간을 떠올려 보세요. 사진 한 장이 3,072개 숫자이고 5만 장을 여러 바퀴 돌리면, CPU로는 수업 시간 안에 끝나지 않습니다. 3주차 표(240행 × 3열)에서는 이 차이가 보이지 않았습니다.

**직접 바꾸기:** 4번의 `Conv2d(3, 32, …)`를 `Conv2d(3, 16, …)`로, `Conv2d(32, 64, …)`를 `Conv2d(16, 32, …)`로 줄이고 `Linear(64 * 8 * 8, 10)`을 `Linear(32 * 8 * 8, 10)`으로 맞춘 뒤 4번부터 다시 실행하세요. 학습할 숫자 개수, 배치당 시간, 정확도가 어떻게 달라지나요? 모델이 작아지면 빨라지지만 정확도는 보통 조금 떨어집니다.

### 공식 참고 문서

- [CIFAR-10 데이터셋](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html)
- [Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [RNN](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html)
- [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- [학습 반복문과 optimizer](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)